In [1]:
# Import data DataFrame from data_preprocessing.ipynb via %run magic
%run ./data_preprocessing.ipynb

int64
int64
Number of (store_id, item_id) groups with entirely missing sell_price: 0
Remaining nulls in sell_price: 0
Number of (store_id, item_id) groups with entirely missing sell_price: 0
Number of rows in data: 5,832,737
Cached 5,747,365 rows to /Users/deepakjacob/projects/Inventory-Forecasting-and-Modeling/data/collections/train_data.csv
<class 'pandas.core.frame.DataFrame'>
Index: 5747365 entries, 28 to 5832736
Data columns (total 29 columns):
 #   Column           Dtype         
---  ------           -----         
 0   id               object        
 1   item_id          object        
 2   dept_id          object        
 3   cat_id           object        
 4   store_id         object        
 5   state_id         object        
 6   d                object        
 7   sales            int16         
 8   date             datetime64[ns]
 9   wm_yr_wk         int64         
 10  weekday          object        
 11  wday             int64         
 12  month            int64 

In [2]:
# Encode categorical variables (category dtype for LightGBM)
cat_cols = ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'event_name_1']

for col in cat_cols:
    data[col] = data[col].astype('category')

In [3]:
# Train / Validation / Test split by date
# Train: up to 2016-02-28  |  Validation: 2016-03-01 to 2016-03-28 (tune here)  |  Test: final 28 days (evaluate here)
drop_cols = ['sales', 'date', 'id', 'd']

train = data[data['date'] <= '2016-02-28']
valid = data[(data['date'] >= '2016-03-01') & (data['date'] <= '2016-03-28')]
test_end = data['date'].max()
test_start = test_end - pd.Timedelta(days=27)
test = data[(data['date'] >= test_start) & (data['date'] <= test_end)]

X_train = train.drop(columns=drop_cols)
y_train = train['sales']
X_valid = valid.drop(columns=drop_cols)
y_valid = valid['sales']
X_test = test.drop(columns=drop_cols)
y_test = test['sales']

print(f"Train:      {train['date'].min().date()} to {train['date'].max().date()}  ({len(train):,} rows)")
print(f"Validation: {valid['date'].min().date()} to {valid['date'].max().date()}  ({len(valid):,} rows) — tune here")
print(f"Test:       {test['date'].min().date()} to {test['date'].max().date()}  ({len(test):,} rows) — final evaluation")

Train:      2011-02-26 to 2016-02-28  (5,576,621 rows)
Validation: 2016-03-01 to 2016-03-28  (85,372 rows) — tune here
Test:       2016-03-28 to 2016-04-24  (85,372 rows) — final evaluation


In [11]:
categorical_features = [
    'item_id',
    'dept_id',
    'cat_id',
    'store_id',
    'state_id',
    'event_name_1',
    'event_type_1',
    'event_name_2',
    'event_type_2',
    'weekday'
]

for col in categorical_features:
    X_train[col] = X_train[col].astype('category')
    X_valid[col] = X_valid[col].astype('category')
    X_test[col] = X_test[col].astype('category')

In [13]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)

In [17]:
from sklearn.preprocessing import OrdinalEncoder

# Encode categorical features
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_train_enc = X_train.copy()
X_valid_enc = X_valid.copy()
X_test_enc = X_test.copy()

X_train_enc[categorical_features] = encoder.fit_transform(X_train[categorical_features])
X_valid_enc[categorical_features] = encoder.transform(X_valid[categorical_features])
X_test_enc[categorical_features] = encoder.transform(X_test[categorical_features])

rf_model.fit(X_train_enc, y_train)

,n_estimators,200
,criterion,'squared_error'
,max_depth,20
,min_samples_split,10
,min_samples_leaf,5
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [22]:
rf_preds = rf_model.predict(X_valid_enc)

# Calculate RMSE
import numpy as np
from sklearn.metrics import mean_squared_error
rmse = np.sqrt(mean_squared_error(y_valid, rf_preds))
print("Validation RMSE:", rmse)



Validation RMSE: 1.7125533574561147


In [25]:
# Make predictions on the test set
test_preds = rf_model.predict(X_test_enc)

# Calculate R^2 score (accuracy/statistic for regression)
from sklearn.metrics import r2_score
test_r2 = r2_score(y_test, test_preds)
from sklearn.metrics import mean_squared_error, mean_absolute_error

test_mse = mean_squared_error(y_test, test_preds)
test_rmse = np.sqrt(test_mse)
test_mae = mean_absolute_error(y_test, test_preds)

print("Test R^2 Score:", test_r2)
print("Test MSE:", test_mse)
print("Test RMSE:", test_rmse)
print("Test MAE:", test_mae)

Test R^2 Score: 0.7633584620053501
Test MSE: 2.763680459297517
Test RMSE: 1.6624320916348785
Test MAE: 0.838629810969394


In [26]:
import pandas as pd

feature_importance = pd.Series(
    rf_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(feature_importance.head(15))

rolling_mean_7     0.864721
rolling_std_7      0.033346
wday               0.019574
rolling_mean_14    0.010521
rolling_mean_28    0.009557
lag_7              0.008231
lag_28             0.008140
wm_yr_wk           0.007736
lag_14             0.007192
item_id            0.006773
sell_price         0.006032
month              0.004873
store_id           0.003381
weekday            0.002922
snap_WI            0.001297
dtype: float64
